# 02 · Features, vendedor y título

El baseline (`01_baseline.ipynb`) llega a **84,2%** usando solo 16 campos que se leen directo del JSON. Este notebook agrega información en tres pasos y mide cuánto aporta cada uno:

1. **Más campos del JSON**: medios de pago, tags, fechas, palabras clave del título y de la garantía.
2. **Historial del vendedor y de la categoría** (*target encoding*).
3. **Lo que dice el título**, con un modelo de texto (TF-IDF + regresión logística).

Todo se evalúa con la misma validación cruzada de 5 particiones que el baseline, así los números son comparables.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score
from lightgbm import LGBMClassifier

# keep_default_dates=False: que pandas NO convierta solo las columnas que terminan en "_time"
train = pd.read_json("data/train_data.jsonlines", lines=True, keep_default_dates=False)
test  = pd.read_json("data/test_data.jsonlines",  lines=True, keep_default_dates=False)

y = (train["condition"] == "used").astype(int).values   # 1 = usado, 0 = nuevo

# train y test juntos, solo para calcular conteos (cuántos ítems tiene cada vendedor, etc.)
todo = pd.concat([train.drop(columns="condition"), test], ignore_index=True)
print(train.shape, test.shape, todo.shape)

## 1. Features

`armar_features` son las 16 del baseline, sin cambios. `armar_features_extra` agrega el resto:

- **Medios de pago fuera de MercadoPago**: cuántos acepta y cuáles (transferencia, efectivo, etc.). Un particular que vende algo usado suele aceptar "acordar con el vendedor"; una tienda, tarjeta y transferencia.
- **Tags** de la publicación y otros indicadores (si tiene video, tienda oficial, precio original tachado).
- **Fechas**: cuándo se creó, cuánto dura la publicación, cuánto pasó entre creación y última actualización.
- **Palabras clave** en el título (`usad`, `nuev`, `impecable`...) y en el texto de garantía (`sin garantía`, `de fábrica`...).
- **Volumen**: cuántos ítems publica el vendedor y cuántos hay en la categoría. Se cuentan sobre train + test juntos; es válido porque no usa la respuesta, solo cuenta apariciones.
- `seller_id` y `category_id` **no entran directo al modelo**: son identificadores con miles de valores distintos. Se usan en la sección 2.

In [ ]:
def g(d, k):
    # lee una clave de un diccionario sin romper si el campo viene vacío
    return d.get(k) if isinstance(d, dict) else None

def armar_features(df):
    X = pd.DataFrame(index=df.index)
    for c in ["price", "base_price", "initial_quantity", "sold_quantity", "available_quantity"]:
        X[c] = df[c]
    X["accepts_mercadopago"] = df["accepts_mercadopago"].astype(int)
    X["automatic_relist"]    = df["automatic_relist"].astype(int)
    X["free_shipping"] = df["shipping"].apply(lambda s: int(bool(g(s, "free_shipping"))))
    X["local_pick_up"] = df["shipping"].apply(lambda s: int(bool(g(s, "local_pick_up"))))
    X["n_pictures"]    = df["pictures"].apply(len)
    X["n_variations"]  = df["variations"].apply(len)
    X["n_attributes"]  = df["attributes"].apply(len)
    X["has_warranty"]  = df["warranty"].notna().astype(int)
    X["listing_type_id"] = df["listing_type_id"]
    X["buying_mode"]     = df["buying_mode"]
    X["shipping_mode"]   = df["shipping"].apply(lambda s: g(s, "mode"))
    return X

def armar_features_extra(df):
    X = pd.DataFrame(index=df.index)

    # medios de pago fuera de MercadoPago
    X["n_pagos"] = df["non_mercado_pago_payment_methods"].apply(len)
    pagos = df["non_mercado_pago_payment_methods"].apply(lambda L: {d.get("id") for d in L})
    for p in ["MLATB", "MLAWC", "MLAMO", "MLAOT", "MLAMC", "MLAVE", "MLAMS"]:
        X["pago_" + p] = pagos.apply(lambda s: int(p in s))

    # tags y otros indicadores
    X["n_tags"] = df["tags"].apply(len)
    for t in ["dragged_bids_and_visits", "good_quality_thumbnail", "dragged_visits",
              "free_relist", "poor_quality_thumbnail"]:
        X["tag_" + t] = df["tags"].apply(lambda L: int(t in L))
    X["tiene_sub_status"]     = df["sub_status"].apply(len)
    X["tiene_video"]          = df["video_id"].notna().astype(int)
    X["tiene_tienda_oficial"] = df["official_store_id"].notna().astype(int)
    X["tiene_precio_original"] = df["original_price"].notna().astype(int)
    X["tiene_parent"]         = df["parent_item_id"].notna().astype(int)
    X["n_descripciones"]      = df["descriptions"].apply(len)

    # precio
    X["dif_precio_base"] = df["price"] - df["base_price"]
    X["precio_redondo"]  = (df["price"] % 10 == 0).astype(int)

    # fechas
    X["duracion_dias"] = (df["stop_time"] - df["start_time"]) / 86_400_000   # vienen en milisegundos
    creado = pd.to_datetime(df["date_created"], utc=True)
    actualizado = pd.to_datetime(df["last_updated"], utc=True)
    X["dias_desde_2013"] = (creado - pd.Timestamp("2013-01-01", tz="UTC")).dt.total_seconds() / 86400
    X["horas_hasta_update"] = (actualizado - creado).dt.total_seconds() / 3600

    # palabras clave en el título
    titulo = df["title"].str.lower()
    X["largo_titulo"] = df["title"].str.len()
    for w in ["usad", "nuev", "original", "impecable", "excelente", "estado",
              "garant", "sin uso", "oferta", "import"]:
        X["titulo_" + w.replace(" ", "_")] = titulo.str.contains(w).astype(int)

    # palabras clave en la garantía
    garantia = df["warranty"].fillna("").str.lower()
    X["largo_garantia"]   = garantia.str.len()
    X["garantia_sin"]     = garantia.str.contains("sin garant").astype(int)
    X["garantia_nuevo"]   = garantia.str.contains("nuev").astype(int)
    X["garantia_fabrica"] = garantia.str.contains("fabric").astype(int)

    # ubicación y volumen
    X["state"] = df["seller_address"].apply(lambda a: g(g(a, "state"), "name"))
    X["vendedor_n_items"]  = df["seller_id"].map(todo["seller_id"].value_counts()).values
    X["categoria_n_items"] = df["category_id"].map(todo["category_id"].value_counts()).values

    # identificadores: NO van al modelo, se usan para el target encoding
    X["seller_id"]   = df["seller_id"]
    X["category_id"] = df["category_id"]
    return X

X_todo = pd.concat([armar_features(todo), armar_features_extra(todo)], axis=1)
X_train = X_todo.iloc[:len(train)].reset_index(drop=True)
X_test  = X_todo.iloc[len(train):].reset_index(drop=True)

BASE  = list(armar_features(todo.head(1)).columns)
EXTRA = [c for c in armar_features_extra(todo.head(1)).columns if c not in ["seller_id", "category_id"]]
CAT   = ["listing_type_id", "buying_mode", "shipping_mode", "state"]

print(f"{len(BASE)} features base + {len(EXTRA)} extra")

## 2. Historial del vendedor y de la categoría: *target encoding*

**La idea:** reemplazar cada vendedor por **la proporción de usados entre sus otras publicaciones**. Un vendedor que publicó 50 ítems y 48 eran usados tiene 0,96; una tienda que solo vende nuevos, cerca de 0.

**Por qué con cuidado:** si para calcular ese número usara también el propio ítem, el modelo estaría "viendo la respuesta". Por ejemplo, un vendedor con un único ítem usado tendría 1,0, y el modelo aprendería que 1,0 = usado sin haber aprendido nada. Eso es *leakage*: rinde perfecto en entrenamiento y falla en test.

**Cómo se evita:** dentro de train, el valor de cada ítem se calcula con las **otras 4 particiones** (*fuera de fold*), nunca con la propia. Para test se usa todo train, porque ahí la respuesta no se conoce.

**Suavizado:** un vendedor con 1 solo ítem no es confiable. Por eso se mezcla su proporción con el promedio general: con pocos ítems pesa más el promedio, con muchos pesa más su propia historia. `suavizado=10` equivale a sumarle 10 ítems "promedio" a cada vendedor.

Vendedores que no aparecen en train reciben el promedio general.

In [ ]:
def target_encoding(clave_train, y_train, clave_test, suavizado=10):
    prior = y_train.mean()

    def tabla_proporciones(claves, ys):
        g = pd.DataFrame({"k": np.asarray(claves), "y": ys}).groupby("k")["y"].agg(["sum", "count"])
        return (g["sum"] + suavizado * prior) / (g["count"] + suavizado)

    # train: cada fila se codifica con las OTRAS particiones
    enc_train = np.full(len(clave_train), prior)
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    for idx_a, idx_b in kf.split(clave_train, y_train):
        tabla = tabla_proporciones(clave_train.iloc[idx_a], y_train[idx_a])
        enc_train[idx_b] = clave_train.iloc[idx_b].map(tabla).fillna(prior).values

    # test: con todo train
    tabla = tabla_proporciones(clave_train, y_train)
    enc_test = clave_test.map(tabla).fillna(prior).values
    return enc_train, enc_test

## 3. Lo que dice el título: TF-IDF + regresión logística

Las palabras clave de la sección 1 son pocas y elegidas a mano. Este modelo aprende solo qué palabras y pares de palabras indican nuevo o usado:

- **TF-IDF** convierte cada título en un vector: cada palabra (y cada par de palabras seguidas) es una columna, con un peso que sube si la palabra aparece en el título y baja si aparece en casi todos los títulos (porque entonces no distingue nada).
- Una **regresión logística** sobre esos vectores da, para cada título, la **probabilidad de que sea usado**.

Esa probabilidad entra como una feature más al LightGBM. Se calcula **fuera de fold**, por el mismo motivo que el target encoding.

In [ ]:
def prob_titulo(titulos_train, y_train, titulos_test):
    modelo = make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
        LogisticRegression(max_iter=3000, C=4),
    )
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    p_train = cross_val_predict(modelo, titulos_train, y_train, cv=kf, method="predict_proba")[:, 1]
    modelo.fit(titulos_train, y_train)
    p_test = modelo.predict_proba(titulos_test)[:, 1]
    return p_train, p_test

## 4. Validación

Misma partición que el baseline (5 particiones estratificadas, `random_state=42`).

El target encoding y la probabilidad del título se recalculan **adentro de cada partición**, usando solo las 4 de entrenamiento. Si se calcularan una vez sobre todo train antes de partir, la partición de validación ya habría influido en esos números y la nota saldría inflada.

`armar_conjunto` es la función que arma las features finales; se usa igual en la validación y en el modelo final, así ambos hacen exactamente lo mismo.

In [ ]:
PARAMS = dict(n_estimators=500, learning_rate=0.05, num_leaves=63, random_state=42, verbose=-1)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def alinear_categorias(A, B):
    # las categorías de B (validación o test) tienen que ser las mismas que las de A (entrenamiento)
    for c in [c for c in CAT if c in A.columns]:
        cats = pd.Categorical(A[c]).categories
        A[c] = pd.Categorical(A[c], categories=cats)
        B[c] = pd.Categorical(B[c].where(B[c].isin(cats)), categories=cats)
    return A, B

def armar_conjunto(XA, yA, titA, XB, titB, columnas, usar_te, usar_titulo):
    A, B = XA[columnas].copy(), XB[columnas].copy()
    if usar_te:
        for c in ["seller_id", "category_id"]:
            A["te_" + c], B["te_" + c] = target_encoding(XA[c], yA, XB[c])
    if usar_titulo:
        A["prob_titulo"], B["prob_titulo"] = prob_titulo(titA, yA, titB)
    return alinear_categorias(A, B)

def evaluar(columnas, usar_te=False, usar_titulo=False):
    notas = []
    for idx_a, idx_b in cv.split(X_train, y):
        A, B = armar_conjunto(
            X_train.iloc[idx_a], y[idx_a], train["title"].iloc[idx_a],
            X_train.iloc[idx_b],           train["title"].iloc[idx_b],
            columnas, usar_te, usar_titulo,
        )
        modelo = LGBMClassifier(**PARAMS).fit(A, y[idx_a])
        notas.append(accuracy_score(y[idx_b], modelo.predict(B)))
    return np.mean(notas), np.std(notas)

## 5. Experimentos

Cada fila agrega algo sobre la anterior. Tarda unos minutos (el modelo de texto es lo más lento).

In [ ]:
experimentos = [
    ("1. Baseline: 16 features",                    BASE,         False, False),
    ("2. + features extra del JSON",                BASE + EXTRA, False, False),
    ("3. + target encoding vendedor y categoría",   BASE + EXTRA, True,  False),
    ("4. + probabilidad del título (TF-IDF)",       BASE + EXTRA, True,  True),
]

resultados = []
for nombre, cols, te, tit in experimentos:
    media, desvio = evaluar(cols, te, tit)
    resultados.append({"experimento": nombre, "accuracy": round(media, 4), "desvío": round(desvio, 4)})
    print(f"{nombre:45s} {media:.4f} ± {desvio:.4f}")

pd.DataFrame(resultados)

## 6. Modelo con todo y submission

Se entrena con los 70.000 ítems de train y se predice test. El formato que acepta Kaggle: columna `ID` del 1 al 30.000 y `condition` con `new` / `used`.

In [ ]:
import os

A, B = armar_conjunto(
    X_train, y, train["title"],
    X_test,     test["title"],
    BASE + EXTRA, usar_te=True, usar_titulo=True,
)
modelo = LGBMClassifier(**PARAMS).fit(A, y)
pred = modelo.predict(B)

sub = pd.DataFrame({
    "ID": range(1, len(test) + 1),
    "condition": np.where(pred == 1, "used", "new"),
})
os.makedirs("submissions", exist_ok=True)
sub.to_csv("submissions/02_vendedor_titulo.csv", index=False)

print(sub.shape)
print(sub["condition"].value_counts(normalize=True).round(3))
sub.head()